### Import

In [1]:
import os
import sys
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt 
from sklearn.model_selection import train_test_split

### General parameters

In [2]:
path_out    = "../Data/EHR/"
path_cohort = "../Data/Cohorts/"
path_timeseries = "Extraction/MIMICIII/Data/Output/"

### Reading Demographic Data

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'admission.csv')
    all_filenames.append(df_file)
    
df_admission = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [4]:
df_admission.head(3)

In [5]:
df_admission.shape

(59801, 18)

### Reading Comorbidities Data

In [5]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'comorbidity.csv')
    all_filenames.append(df_file)
    
df_comorbidity = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [6]:
df_comorbidity.head(3)

### Reading Missing Percentage

In [7]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'missing_percentage.csv')
    all_filenames.append(df_file)
    
df_missing = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])

In [8]:
df_missing.head(3)

### Reading Data

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in (all_stays):
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
    all_filenames.append(df_file)
    
df = pd.concat([pd.read_csv(file, nrows=60, low_memory=False) for file in all_filenames if os.path.exists(file)])
df = df[(df.Bins >= 0) & (df.Bins <= 23)]
df = df.reset_index(drop=True)

In [4]:
df.head(3)

### Reading Data based on Hours

In [ ]:
def read_rows_startH_to_endH(path_timeseries, start=24, length=24, start_h=24, end_h=47):
    
    all_stays  = pd.Series(os.listdir(path_timeseries))
    all_filenames = []

    for stay_id in (all_stays):
        df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
        all_filenames.append(df_file)

    dfs = []
    for file in all_filenames:
        if os.path.exists(file):
            df = pd.read_csv(file, skiprows=range(1, start), nrows=length, low_memory=False)
            df = df[(df.Bins >= start_h) & (df.Bins <= end_h)]
            if not df.empty:
                dfs.append(df)

    final_df = pd.concat(dfs, ignore_index=True)
    return final_df

In [ ]:
second_day_df = read_rows_startH_to_endH(path_timeseries, start=24, length=48, start_h=24, end_h=47)
second_day_df = second_day_df.reset_index(drop=True)

In [ ]:
third_day_df = read_rows_startH_to_endH(path_timeseries, start=48, length=48, start_h=48, end_h=71)
third_day_df = third_day_df.reset_index(drop=True)

### Read Notes ID

In [3]:
note_columns = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME', 'OUTTIME', 'Bins', 
                'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'Note']

In [4]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.csv')
    all_filenames.append(df_file)
    
df_note_ids = pd.concat([pd.read_csv(file, low_memory=False, usecols=note_columns) for file in all_filenames if os.path.exists(file)])

In [5]:
df_note_ids = df_note_ids[df_note_ids.Note.notnull()]
df_note_ids = df_note_ids.drop_duplicates()
df_note_ids = df_note_ids.reset_index(drop=True)

df_note_ids = df_note_ids[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'INTIME', 'OUTTIME', 'Bins', 
                           'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG', 'Note']]

In [6]:
df_note_ids.head(3)

### Missing in 24H

In [ ]:
missing_value_df = df.groupby('stay_id').apply(lambda x: x.isnull().all())
missing_value_df.drop(columns=['stay_id'], inplace=True)
missing_value_df = missing_value_df.reset_index()
missing_value_df = missing_value_df.replace(True, np.nan)
percent_missing  = missing_value_df.isnull().sum() * 100 / len(missing_value_df)
missing_value_perc = pd.DataFrame({'column_name': missing_value_df.columns, 'percent_missing': percent_missing})
missing_value_perc.sort_values('percent_missing', inplace=True, ascending=False)
missing_value_perc.reset_index(inplace=True, drop=True)
missing_value_perc.head()

### Save Data

In [11]:
df_admission.to_csv(path_out + 'demographic_all.csv', index=False)
df_comorbidity.to_csv(path_out + 'comorbidity_all.csv', index=False)
df_missing.to_csv(path_out + 'missing_all.csv', index=False)
df.to_csv(path_out + '0h_to_24h_data.csv', index=False)
df_note_ids.to_csv(path_out + 'all_note_ids.csv', index=False)